In [1]:
import os
import json
import re
from collections import defaultdict
from typing import List, Dict, Tuple, Union
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [2]:
class CustomByteLevelBPE:
    RESERVED_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
    END_OF_WORD = "</w>"

    @staticmethod
    def bytes_to_unicode():
        byte_list = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
        char_list = byte_list[:]
        offset = 0
        for byte_val in range(2**8):
            if byte_val not in byte_list:
                byte_list.append(byte_val)
                char_list.append(2**8 + offset)
                offset += 1
        char_list = [chr(c) for c in char_list]
        return dict(zip(byte_list, char_list))

    def __init__(self, vocab_size: int = 30000):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.inv_vocab = {}
        self.merges = []
        self.merge_ranks = {}

        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3

        self.byte_encoder = self.bytes_to_unicode()
        self.byte_decoder = {char: byte_val for byte_val, char in self.byte_encoder.items()}

        self._init_reserved_vocab()

    def _init_reserved_vocab(self):
        self.vocab = {}
        self.inv_vocab = {}
        for token_idx, token in enumerate(self.RESERVED_TOKENS):
            self.vocab[token] = token_idx
            self.inv_vocab[token_idx] = token

    def _byte_to_symbol(self, byte_val: int) -> str:
        return self.byte_encoder[byte_val]

    def _symbol_to_bytes(self, symbol: str) -> list[int]:
        return [self.byte_decoder[char] for char in symbol]

    def train(self, text_corpus: list[str]):
        self._init_reserved_vocab()
        self.merges = []
        self.merge_ranks = {}

        token_to_id = dict(self.vocab)
        id_to_token = dict(self.inv_vocab)

        def get_or_register_token(symbol: str) -> int:
            if symbol not in token_to_id:
                new_id = len(token_to_id)
                token_to_id[symbol] = new_id
                id_to_token[new_id] = symbol
            return token_to_id[symbol]

        base_symbols = sorted([self._byte_to_symbol(b) for b in range(256)] + [self.END_OF_WORD])
        for symbol in base_symbols:
            get_or_register_token(symbol)

        eow_id = token_to_id[self.END_OF_WORD]

        word_frequencies = defaultdict(int)
        for document in text_corpus:
            for word in document.split():
                if not word:
                    continue
                raw_bytes = list(word.encode("utf-8"))
                word_token_ids = tuple([token_to_id[self._byte_to_symbol(b)] for b in raw_bytes] + [eow_id])
                word_frequencies[word_token_ids] += 1

        distinct_ids = set()
        for word_token_ids in word_frequencies:
            distinct_ids.update(word_token_ids)
        for token_id in sorted(distinct_ids):
            symbol = id_to_token[token_id]
            if symbol not in self.vocab:
                self.vocab[symbol] = len(self.vocab)
                self.inv_vocab[self.vocab[symbol]] = symbol

        print(f"Initial base vocab size (including 4 reserved tokens): {len(self.vocab)}", flush=True)

        pair_counts = defaultdict(int)
        tokenized_corpus = dict(word_frequencies)
        for word_tuple, freq in tokenized_corpus.items():
            for i in range(len(word_tuple) - 1):
                pair_counts[(word_tuple[i], word_tuple[i + 1])] += freq

        progress_bar = tqdm(total=self.vocab_size, initial=len(self.vocab), desc="Training BPE Progress")

        while len(self.vocab) < self.vocab_size:
            if not pair_counts:
                break

            max_frequency = max(pair_counts.values())
            if max_frequency < 2:
                break

            frequent_pairs = [pair for pair, count in pair_counts.items() if count == max_frequency]
            best_pair_ids = sorted(frequent_pairs, key=lambda pair: (id_to_token[pair[0]], id_to_token[pair[1]]))[0]
            first_id, second_id = best_pair_ids

            merged_symbol = id_to_token[first_id] + id_to_token[second_id]
            merged_id = get_or_register_token(merged_symbol)
            self.vocab[merged_symbol] = len(self.vocab)
            self.inv_vocab[self.vocab[merged_symbol]] = merged_symbol
            token_to_id[merged_symbol] = merged_id
            id_to_token[merged_id] = merged_symbol

            best_pair_symbols = (id_to_token[first_id], id_to_token[second_id])
            self.merges.append(best_pair_symbols)
            self.merge_ranks[best_pair_symbols] = len(self.merges) - 1

            updated_corpus = {}
            for word_tuple, freq in tokenized_corpus.items():
                if first_id not in word_tuple:
                    updated_corpus[word_tuple] = freq
                    continue

                for i in range(len(word_tuple) - 1):
                    pair = (word_tuple[i], word_tuple[i + 1])
                    updated_count = pair_counts.get(pair, 0) - freq
                    if updated_count <= 0:
                        pair_counts.pop(pair, None)
                    else:
                        pair_counts[pair] = updated_count

                merged_word = []
                idx = 0
                while idx < len(word_tuple):
                    if idx < len(word_tuple) - 1 and word_tuple[idx] == first_id and word_tuple[idx + 1] == second_id:
                        merged_word.append(merged_id)
                        idx += 2
                    else:
                        merged_word.append(word_tuple[idx])
                        idx += 1
                merged_tuple = tuple(merged_word)
                updated_corpus[merged_tuple] = freq

                for i in range(len(merged_tuple) - 1):
                    pair_counts[(merged_tuple[i], merged_tuple[i + 1])] += freq

            tokenized_corpus = updated_corpus

            progress_bar.update(1)
            if len(self.vocab) % 1000 == 0:
                print(f"BPE Progress: {len(self.vocab):,} / {self.vocab_size:,} tokens ({(len(self.vocab)/self.vocab_size)*100:.1f}%)", flush=True)

        progress_bar.close()
        print(f"Training complete! Final vocabulary size: {len(self.vocab)}", flush=True)

    def _encode_word(self, word: str) -> list[int]:
        raw_bytes = list(word.encode("utf-8"))
        symbols = [self._byte_to_symbol(b) for b in raw_bytes] + [self.END_OF_WORD]

        while len(symbols) > 1:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            valid_pairs = [pair for pair in pairs if pair in self.merge_ranks]
            if not valid_pairs:
                break

            best_pair = min(valid_pairs, key=lambda pair: self.merge_ranks[pair])

            merged_symbols = []
            idx = 0
            replacement = best_pair[0] + best_pair[1]
            while idx < len(symbols):
                if idx < len(symbols) - 1 and (symbols[idx], symbols[idx + 1]) == best_pair:
                    merged_symbols.append(replacement)
                    idx += 2
                else:
                    merged_symbols.append(symbols[idx])
                    idx += 1
            symbols = merged_symbols

        token_ids = []
        for symbol in symbols:
            token_ids.append(self.vocab.get(symbol, self.unk_token_id))
        return token_ids

    @classmethod
    def from_pretrained_hub(cls, repo_id: str, token: str = None):
        vocab_path = hf_hub_download(repo_id=repo_id, filename="vocab.json", repo_type="model", token=token)
        hf_hub_download(repo_id=repo_id, filename="merges.txt", repo_type="model", token=token)
        model_directory = os.path.dirname(vocab_path)
        tokenizer = cls()
        tokenizer.load(model_directory)
        return tokenizer

    def encode(self, text: str, add_special_tokens: bool = False) -> list[int]:
        words = text.split()
        token_ids = []
        if add_special_tokens:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(self._encode_word(word))

        if add_special_tokens:
            token_ids.append(self.eos_token_id)

        return token_ids

    def load(self, folder_path: str):
        with open(os.path.join(folder_path, "vocab.json"), "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        self.inv_vocab = {token_id: symbol for symbol, token_id in self.vocab.items()}

        self.merges = []
        with open(os.path.join(folder_path, "merges.txt"), "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#version"):
                    continue
                tokens = line.split(" ")
                if len(tokens) == 2:
                    self.merges.append((tokens[0], tokens[1]))

        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.vocab_size = len(self.vocab)
        print(f"Loaded custom BPE tokenizer from {folder_path} (vocab size: {self.vocab_size})")

    def save(self, folder_path: str):
        os.makedirs(folder_path, exist_ok=True)
        with open(os.path.join(folder_path, "vocab.json"), "w", encoding="utf-8") as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)
        with open(os.path.join(folder_path, "merges.txt"), "w", encoding="utf-8") as f:
            f.write("#version: 0.2\n")
            for pair in self.merges:
                f.write(f"{pair[0]} {pair[1]}\n")
        print(f"Saved custom BPE tokenizer to: {folder_path}")

    def decode(self, token_ids: list[int], skip_special_tokens: bool = True) -> str:
        byte_stream = []
        special_token_ids = {self.pad_token_id, self.unk_token_id, self.bos_token_id, self.eos_token_id}
        for token_id in token_ids:
            if skip_special_tokens and token_id in special_token_ids:
                continue
            symbol = self.inv_vocab.get(token_id, "")
            if not symbol:
                continue
            if self.END_OF_WORD in symbol:
                clean_symbol = symbol.replace(self.END_OF_WORD, "")
                byte_stream.extend(self._symbol_to_bytes(clean_symbol))
                byte_stream.append(ord(" "))
            else:
                byte_stream.extend(self._symbol_to_bytes(symbol))

        return bytes(byte_stream).decode("utf-8", errors="replace").strip()

    def push_to_hub(self, repo_id: str, token: str):
        temp_dir = "/tmp/bpe_temp"
        self.save(temp_dir)
        api = HfApi()
        api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=token)
        api.upload_folder(folder_path=temp_dir, repo_id=repo_id, repo_type="model", token=token)
        print(f"Successfully pushed tokenizer to Hugging Face Hub: {repo_id}")

    def __len__(self):
        return len(self.vocab)

print("Class defined.")

Class defined.


In [3]:
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF")
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"

target_total_tokens = 5_000_000
vocab_size = 30000

data_ratios = {
    "Phase_1_Pretraining/Train/train_acts.json": 0.10,
    "Phase_1_Pretraining/Train/train_supreme_court.json": 0.45,
    "Phase_1_Pretraining/Train/train_high_courts.json": 0.45,
    "Phase_1_Pretraining/Train/train_tribunals.json": 0.10,
}

hf_tokenizer_repo = "Granth2/cs781-custom-bpe-30k-Version-10-40-40-10"

def stream_json_array(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        char = ""
        while char != "[":
            char = file.read(1)
            if not char:
                return
        decoder = json.JSONDecoder()
        buffer = ""
        for line in file:
            buffer += line
            while True:
                buffer = buffer.lstrip()
                if not buffer:
                    break
                if buffer[0] == ",":
                    buffer = buffer[1:].lstrip()
                    if not buffer:
                        break
                if buffer[0] == "]":
                    return
                try:
                    record, end_idx = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[end_idx:]
                except json.JSONDecodeError:
                    break

sampled_corpus = []
print("Starting proportional dataset sampling:")

for file_name, ratio in data_ratios.items():
    target_category_tokens = int(target_total_tokens * ratio)
    print(f"\nProcessing {file_name} | Target Ratio: {ratio * 100:.0f}% ({target_category_tokens:,} tokens)", flush=True)

    try:
        downloaded_file = hf_hub_download(
            repo_id=hf_dataset_repo,
            filename=file_name,
            repo_type="dataset",
            token=hf_token
        )

        category_tokens_collected = 0
        category_docs_added = 0

        for record in stream_json_array(downloaded_file):
            document_text = record["text"]
            estimated_tokens = len(document_text.split()) * 1.25

            sampled_corpus.append(document_text)
            category_tokens_collected += estimated_tokens
            category_docs_added += 1

            if category_tokens_collected >= target_category_tokens:
                break

        print(f"--> Collected {category_docs_added:,} documents (~{int(category_tokens_collected):,} tokens) for {file_name}", flush=True)

    except Exception as error:
        print(f"Error processing {file_name}: {error}", flush=True)

print(f"\nTotal sampled dataset ready! Total documents: {len(sampled_corpus):,}", flush=True)

tokenizer = CustomByteLevelBPE(vocab_size=vocab_size)

print(f"\nTraining Custom BPE Tokenizer (Vocab Size: {vocab_size:,})...", flush=True)
tokenizer.train(sampled_corpus)

tokenizer.save("./bpe_model_30k")

tokenizer.push_to_hub(repo_id=hf_tokenizer_repo, token=hf_token)
print("Execution Completed Successfully!", flush=True)

Starting proportional dataset sampling:

Processing Phase_1_Pretraining/Train/train_acts.json | Target Ratio: 10% (500,000 tokens)


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

--> Collected 62 documents (~523,817 tokens) for Phase_1_Pretraining/Train/train_acts.json

Processing Phase_1_Pretraining/Train/train_supreme_court.json | Target Ratio: 45% (2,250,000 tokens)


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

--> Collected 379 documents (~2,252,570 tokens) for Phase_1_Pretraining/Train/train_supreme_court.json

Processing Phase_1_Pretraining/Train/train_high_courts.json | Target Ratio: 45% (2,250,000 tokens)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

--> Collected 3,710 documents (~2,250,147 tokens) for Phase_1_Pretraining/Train/train_high_courts.json

Processing Phase_1_Pretraining/Train/train_tribunals.json | Target Ratio: 10% (500,000 tokens)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

--> Collected 80 documents (~508,841 tokens) for Phase_1_Pretraining/Train/train_tribunals.json

Total sampled dataset ready! Total documents: 4,231

Training Custom BPE Tokenizer (Vocab Size: 30,000)...
Initial base vocab size (including 4 reserved tokens): 158


Training BPE Progress:   1%|          | 158/30000 [00:00<?, ?it/s]

BPE Progress: 1,000 / 30,000 tokens (3.3%)
BPE Progress: 2,000 / 30,000 tokens (6.7%)
BPE Progress: 3,000 / 30,000 tokens (10.0%)
BPE Progress: 4,000 / 30,000 tokens (13.3%)
BPE Progress: 5,000 / 30,000 tokens (16.7%)
BPE Progress: 6,000 / 30,000 tokens (20.0%)
BPE Progress: 7,000 / 30,000 tokens (23.3%)
BPE Progress: 8,000 / 30,000 tokens (26.7%)
BPE Progress: 9,000 / 30,000 tokens (30.0%)
BPE Progress: 10,000 / 30,000 tokens (33.3%)
BPE Progress: 11,000 / 30,000 tokens (36.7%)
BPE Progress: 12,000 / 30,000 tokens (40.0%)
BPE Progress: 13,000 / 30,000 tokens (43.3%)
BPE Progress: 14,000 / 30,000 tokens (46.7%)
BPE Progress: 15,000 / 30,000 tokens (50.0%)
BPE Progress: 16,000 / 30,000 tokens (53.3%)
BPE Progress: 17,000 / 30,000 tokens (56.7%)
BPE Progress: 18,000 / 30,000 tokens (60.0%)
BPE Progress: 19,000 / 30,000 tokens (63.3%)
BPE Progress: 20,000 / 30,000 tokens (66.7%)
BPE Progress: 21,000 / 30,000 tokens (70.0%)
BPE Progress: 22,000 / 30,000 tokens (73.3%)
BPE Progress: 23,000 